In [30]:
import gymnasium as gym
import numpy as np
import pandas as pd
from gymnasium import spaces

from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import CallbackList
from stable_baselines3 import PPO

from importlib import reload
import fitness_functions
reload(fitness_functions)
from fitness_functions import fitness_ESM, fitness_ESM_DMS
import callbacks
reload(callbacks)
from callbacks import *
import environments
reload(environments)
from environments import ProteinEnv

In [ ]:
import pickle, torch
import torch.nn as nn
with open('aav_embeddings_wrong.pkl', 'rb') as file:
    emb = pickle.load(file)

DMS = pd.read_csv('aav_dms.csv')

In [ ]:
with open('aav_wt.txt', 'r') as file:
    wt = file.readline().strip()

def make_env():
    # Provide your own initial sequence + fitness_fn
    return ProteinEnv(wt, fitness_ESM_DMS, 'aav_dms.csv')

vec_env = DummyVecEnv([make_env])

model = PPO(
    policy="MlpPolicy",
    env=vec_env,
    learning_rate=3e-4,
    n_steps=3, 
    batch_size=1,
    gae_lambda=0.95,
    gamma=0.99,
    n_epochs=1,
    clip_range=0.2,
    verbose=1,
    device="cpu"
)

total_timesteps = 64
tqdm_cb = TQDMCallback(total_timesteps=total_timesteps, algo='PPO')
logger_cb = ProteinRLLogger(check_freq=1)
callback = CallbackList([tqdm_cb, logger_cb])

import cProfile
import pstats

profiler = cProfile.Profile()
profiler.enable()
model.learn(total_timesteps=total_timesteps, callback=callback)

profiler.disable()
stats = pstats.Stats(profiler)
stats.sort_stats('cumulative')
stats.print_stats(20)  # Top 20 time consumers

model.save("ppo_pretraining")

Exception ignored When destroying _lsprof profiler:
Traceback (most recent call last):
  File "/tmp/ipykernel_1914796/2990426909.py", line 33, in <module>
RuntimeError: Cannot install a profile function while another profile function is being installed


Using cpu device


Pos: 565 and aa_idx: 7


Used DMS
Pos: 476 and aa_idx: 0


Used surrogate
Pos: 412 and aa_idx: 18


...

Used DMS
Pos: 215 and aa_idx: 13


Used surrogate
Pos: 306 and aa_idx: 5


Used surrogate
Pos: 358 and aa_idx: 16


Used surrogate
Pos: 570 and aa_idx: 9


KeyboardInterrupt: 

In [2]:
import pickle
with open('debug.pkl', 'rb') as file:
    debug = pickle.load(file)

all = debug['all_rewards']
mut = debug['mut']

In [6]:
sum(mut)

np.float64(89.0)

In [16]:
def levenshtein_distance(s, t):
    m, n = len(s), len(t)
    d = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        d[i][0] = i
    for j in range(n + 1):
        d[0][j] = j

    for j in range(1, n + 1):
        for i in range(1, m + 1):
            cost = 0 if s[i - 1] == t[j - 1] else 1
            d[i][j] = min(d[i - 1][j] + 1,      # Deletion
                          d[i][j - 1] + 1,      # Insertion
                          d[i - 1][j - 1] + cost) # Substitution

    return d[m][n]

string1 = wt
string2 = "MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLVLPGYKYLGPFNGADKGEPVNEADAAALEHDKAYDRQLDSGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRVLEPLGLVEEPVKTAPGKKRPVEHSPVEPDSSSGTGKAGQQPARKRLNFGQTGDADSVPDPQPLGQPPAAPSGLGTNTMATGSGAPMADNNEGADGVGNSSGNWHCDSTWMGDRVITTSTRFWALPTYNNHLYKQISSQSGASNDNHYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTQNDGTTTIANNLTSTVQVFTDSEYQLPYVLGSAHQGCLPPFPADVFMVPQYGYLTLNNGSQAVGRSSFYCLEYFPSQMLRTGNNFTFSYTFEDVPFHSSYAHSQSLDRLMNPLIDQVLYYLSRTNTPSGTTTQSRLQFSQAGASDIRDQSRNWLPGPCYRQQRVSKTSADNNNSEYSWTGATKYHLNGRDSLVNPGPAMASHKDDEEKFFPQSGVLIFGKQGSEKTNVDIEKVMITDEEEIRTTNPVATEQYGSVSTNLQRGNRQAATADVNTQGVLPGMVWQDRDVYLQGPIWAKIPHTDGHFHPSPLMGGFGLKHPPPQILIKNTPVPANPSTTFSAAKFASFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYNKSVNVDFTVDTNGVYSEPRPIGTRYLTRNL"
distance = levenshtein_distance(string1, string2)
print(f"The Levenshtein distance is: {distance}")

The Levenshtein distance is: 3
